# Retreino do `gesture_recognizer.task` (Libras) com Python 3.10

O Colab hoje usa Python 3.12, e o `mediapipe-model-maker` só funciona até o 3.10. Este notebook cria um **Python 3.10 separado** dentro do Colab (com o `uv`) e roda o Model Maker nele, com versões fixas que foram testadas juntas.

**O que mudou em relação ao notebook de 2025** (pontos do Deep Review):
- O teste fica separado: o treino usa só a pasta `train` (90% treino, 10% validação). A pasta `test` só é usada no fim, para avaliar.
- Lote de 32 fotos (antes era 2, o que deixava o BatchNorm instável) e 100 épocas (antes 10).
- Espelhamento das fotos de treino (mão esquerda e direita), igual ao RF, SVM e SNN.
- Sementes fixas (42) e lista das fotos usadas, para dar para repetir.

**Como usar:** `Ambiente de execução > Executar tudo`. No fim ele baixa `treino_task_libras.zip` com o `.task`, os números e os gráficos.

Tempo estimado com as 46 mil fotos e espelhamento: cerca de 1 a 2 horas (a parte lenta é achar a mão em cada foto). Não precisa de GPU.

## 1. Configuração

In [ ]:
# De onde vêm as fotos:
#   "kaggle" -> baixa a base williansoliveira/libras direto do Kaggle (46.262 fotos). Não precisa do Drive.
#   "drive"  -> usa o libras.zip que está no seu Google Drive (caminho em ZIP_DRIVE). Vai pedir permissão do Drive.
#   "zip"    -> usa um libras.zip que você sobe no painel de arquivos do Colab (à esquerda).
FONTE = "kaggle"
ZIP_DRIVE = "/content/drive/MyDrive/libras.zip"   # ajuste se o zip estiver em outra pasta do Drive
FRACAO = 1.0      # 1.0 = base inteira; 0.1 = cerca de 10% de cada letra (sorteio com semente 42)
ESPELHAR = True   # acrescenta a versão espelhada de cada foto de treino
EPOCAS = 100
LOTE = 32
SEMENTE = 42

## 2. Python 3.10 com o Model Maker (leva uns 2 minutos)

In [ ]:
!pip -q install uv
!uv python install 3.10
!rm -rf /content/mm310 && uv venv -q -p 3.10 /content/mm310
# Passo 1: Model Maker e TensorFlow 2.15 (versões testadas juntas)
!uv pip install -q -p /content/mm310/bin/python \
    "mediapipe-model-maker==0.2.1.4" "mediapipe==0.10.14" \
    "tensorflow==2.15.1" "tf-models-official==2.15.0" "tensorflow-text==2.15.0" \
    "tensorflow-addons==0.23.0" "numpy==1.26.4" pillow
# Passo 2 (separado de propósito): volta o protobuf para 3.20.3, senão o import quebra com "runtime_version"
!uv pip install -q -p /content/mm310/bin/python \
    "tensorflow-metadata==1.14.0" "tensorflow-datasets==4.9.3" "setuptools<70"
import os, subprocess
# O Colab define MPLBACKEND para o próprio notebook; o Python 3.10 precisa de "Agg"
ENV310 = {**os.environ, "MPLBACKEND": "Agg"}
teste = subprocess.run(["/content/mm310/bin/python", "-W", "ignore", "-c",
    "import sys, tensorflow as tf, mediapipe as mp; from mediapipe_model_maker import gesture_recognizer; "
    "print('OK  Python', sys.version.split()[0], '| TensorFlow', tf.__version__, '| MediaPipe', mp.__version__)"],
    capture_output=True, text=True, env=ENV310)
if teste.returncode != 0:
    print(teste.stderr[-3000:])
    raise RuntimeError("A instalação do Python 3.10 falhou. Copie a mensagem acima e mande para o Claude.")
print(teste.stdout.strip().splitlines()[-1])

## 3. Fotos

In [ ]:
import os, random, shutil, zipfile

def achar_raiz(base):
    for raiz, pastas, _ in os.walk(base):
        if "train" in pastas and "test" in pastas:
            return raiz
    raise FileNotFoundError("Não achei as pastas train/ e test/ em " + base)

if FONTE == "kaggle":
    !pip -q install kagglehub
    import kagglehub
    raiz = achar_raiz(kagglehub.dataset_download("williansoliveira/libras"))
else:
    if FONTE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        arq_zip = ZIP_DRIVE
    else:
        arq_zip = "/content/libras.zip"
    if not os.path.exists(arq_zip):
        raise FileNotFoundError(f"Não achei {arq_zip}. Confira o caminho e rode esta célula de novo.")
    print("Descompactando", arq_zip, "...")
    shutil.rmtree("/content/libras_zip", ignore_errors=True)
    zipfile.ZipFile(arq_zip).extractall("/content/libras_zip")
    raiz = achar_raiz("/content/libras_zip")
print("Pasta das fotos:", raiz)

# Monta /content/dados com a fração escolhida (links, sem copiar) e grava a lista de fotos usadas
EXT = (".jpg", ".jpeg", ".png")
DADOS = "/content/dados"
shutil.rmtree(DADOS, ignore_errors=True)
rng = random.Random(SEMENTE)
lista = []
for parte in ("train", "test"):
    for letra in sorted(os.listdir(os.path.join(raiz, parte))):
        origem = os.path.join(raiz, parte, letra)
        if not os.path.isdir(origem) or letra.lower() == "none":
            continue
        fotos = sorted(f for f in os.listdir(origem) if f.lower().endswith(EXT))
        if FRACAO < 1.0:
            fotos = sorted(rng.sample(fotos, max(1, round(len(fotos) * FRACAO))))
        destino = os.path.join(DADOS, parte, letra)
        os.makedirs(destino, exist_ok=True)
        for f in fotos:
            os.symlink(os.path.join(origem, f), os.path.join(destino, f))
            lista.append(f"{parte}/{letra}/{f}")
os.makedirs("/content/resultado", exist_ok=True)
with open("/content/resultado/lista_fotos.txt", "w") as f:
    f.write("\n".join(lista))
n_tr = sum(x.startswith("train/") for x in lista)
print(f"Fotos: {len(lista)} (treino {n_tr}, teste {len(lista) - n_tr}) em {len(os.listdir(os.path.join(DADOS, 'train')))} letras")

## 4. Script de treino (roda no Python 3.10)

In [ ]:
%%writefile /content/treinar_task.py
"""Treina o gesture_recognizer.task do MediaPipe Model Maker (roda no Python 3.10).

Diferenças em relação ao notebook de 2025:
  - o teste fica separado: o treino usa só a pasta train (90% treino, 10% validação)
    e a avaliação usa só a pasta test, que o modelo nunca viu;
  - lote de 32 fotos em vez de 2 (o lote de 2 deixava o BatchNorm instável);
  - mais épocas e espelhamento opcional das fotos de treino (mão esquerda/direita).

Uso: python treinar_task.py --dados PASTA --saida PASTA [--epocas 100] [--lote 32] [--espelhar]
A PASTA de dados precisa ter train/<letra>/*.jpg e test/<letra>/*.jpg.
"""
import argparse
import json
import os
import random
import shutil
import time

import numpy as np

p = argparse.ArgumentParser()
p.add_argument("--dados", required=True)
p.add_argument("--saida", required=True)
p.add_argument("--epocas", type=int, default=100)
p.add_argument("--lote", type=int, default=32)
p.add_argument("--lr", type=float, default=0.001)
p.add_argument("--espelhar", action="store_true")
p.add_argument("--semente", type=int, default=42)
a = p.parse_args()

random.seed(a.semente)
np.random.seed(a.semente)
import tensorflow as tf  # noqa: E402

tf.random.set_seed(a.semente)
from mediapipe_model_maker import gesture_recognizer  # noqa: E402
from PIL import Image, ImageOps  # noqa: E402

EXT = (".jpg", ".jpeg", ".png")
trab = os.path.join(a.saida, "_pastas")
shutil.rmtree(trab, ignore_errors=True)
shutil.rmtree(os.path.join(a.saida, "modelo"), ignore_errors=True)  # evita retomar checkpoint antigo


def montar(origem, destino, espelhar):
    """Liga as fotos em destino/<letra> e cria a pasta 'none' vazia que o Model Maker exige."""
    os.makedirs(os.path.join(destino, "none"), exist_ok=True)
    n = 0
    for letra in sorted(os.listdir(origem)):
        d = os.path.join(origem, letra)
        if not os.path.isdir(d) or letra.lower() == "none":
            continue
        out = os.path.join(destino, letra)
        os.makedirs(out, exist_ok=True)
        for f in sorted(os.listdir(d)):
            if not f.lower().endswith(EXT):
                continue
            os.symlink(os.path.abspath(os.path.join(d, f)), os.path.join(out, f))
            n += 1
            if espelhar:
                ImageOps.mirror(Image.open(os.path.join(d, f)).convert("RGB")).save(
                    os.path.join(out, "espelho_" + os.path.splitext(f)[0] + ".jpg"), quality=95)
                n += 1
    return n


t0 = time.time()
n_treino = montar(os.path.join(a.dados, "train"), os.path.join(trab, "train"), a.espelhar)
n_teste = montar(os.path.join(a.dados, "test"), os.path.join(trab, "test"), False)
print(f"Fotos de treino{' (com espelhadas)' if a.espelhar else ''}: {n_treino} | fotos de teste: {n_teste}")

print("Detectando mãos e calculando o resumo de 128 números (é a parte demorada)...")
# Barra de progresso: conta cada foto que o Model Maker abre para achar a mão
from mediapipe_model_maker.python.vision.gesture_recognizer import dataset as _mm_ds  # noqa: E402
_ImgOrig = _mm_ds._Image
_prog = {"n": 0, "total": 0, "t0": 0.0, "etapa": ""}


class _ImgContador:
    @staticmethod
    def create_from_file(caminho):
        _prog["n"] += 1
        n, tot = _prog["n"], _prog["total"]
        if n == 1:
            _prog["t0"] = time.time()
        if n % 500 == 0 or n == tot:
            gasto = time.time() - _prog["t0"]
            falta = gasto / n * (tot - n)
            print(f"  {_prog['etapa']}: {n}/{tot} fotos ({100 * n / tot:.0f}%) | "
                  f"passou {gasto / 60:.0f} min | falta ~{falta / 60:.0f} min", flush=True)
        return _ImgOrig.create_from_file(caminho)


_mm_ds._Image = _ImgContador
_prog.update(n=0, total=n_treino, etapa="Treino")
pre = gesture_recognizer.HandDataPreprocessingParams()
dados = gesture_recognizer.Dataset.from_folder(dirname=os.path.join(trab, "train"), hparams=pre)
treino, validacao = dados.split(0.9)
_prog.update(n=0, total=n_teste, etapa="Teste")
teste = gesture_recognizer.Dataset.from_folder(dirname=os.path.join(trab, "test"), hparams=pre)
print(f"Com mão detectada -> treino: {len(treino)}, validação: {len(validacao)}, teste: {len(teste)}")
t_pre = time.time() - t0

hparams = gesture_recognizer.HParams(
    learning_rate=a.lr, batch_size=a.lote, epochs=a.epocas,
    export_dir=os.path.join(a.saida, "modelo"))
opcoes = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
t1 = time.time()
modelo = gesture_recognizer.GestureRecognizer.create(
    train_data=treino, validation_data=validacao, options=opcoes)
t_treino = time.time() - t1

perda, acc = modelo.evaluate(teste, batch_size=32)
print(f"\nTESTE (pasta test, nunca vista no treino): acurácia {acc:.4f}, perda {perda:.4f}")

# Matriz de confusão e métricas por letra no teste
rotulos = list(teste.label_names)
ds = teste.gen_tf_dataset(batch_size=32, is_training=False)
y_true, y_pred = [], []
for x, y in ds:
    prob = modelo._model.predict(x, verbose=0)
    y_true += list(np.argmax(y.numpy(), 1))
    y_pred += list(np.argmax(prob, 1))
y_true, y_pred = np.array(y_true), np.array(y_pred)
k = len(rotulos)
matriz = np.zeros((k, k), dtype=int)
for t, pr in zip(y_true, y_pred):
    matriz[t, pr] += 1
por_letra = {}
for i, r in enumerate(rotulos):
    vp = matriz[i, i]
    sup = matriz[i].sum()
    if sup == 0:
        continue
    prec = vp / matriz[:, i].sum() if matriz[:, i].sum() else 0.0
    rev = vp / sup
    f1 = 2 * prec * rev / (prec + rev) if prec + rev else 0.0
    por_letra[r] = {"precisao": round(float(prec), 4), "revocacao": round(float(rev), 4),
                    "f1": round(float(f1), 4), "fotos": int(sup)}
pesos = np.array([v["fotos"] for v in por_letra.values()])
media = {m: round(float(np.average([v[m] for v in por_letra.values()], weights=pesos)), 4)
         for m in ("precisao", "revocacao", "f1")}
print("Médias ponderadas no teste:", media)
for r, v in por_letra.items():
    print(f"  {r}: P={v['precisao']:.2f} R={v['revocacao']:.2f} F1={v['f1']:.2f} ({v['fotos']} fotos)")

modelo.export_model()
hist = {k2: [float(x) for x in v] for k2, v in modelo._history.history.items()}
res = {
    "config": {"epocas": a.epocas, "lote": a.lote, "lr": a.lr, "espelhar": a.espelhar,
               "semente": a.semente, "divisao": "train -> 90% treino / 10% validação; test só para avaliar"},
    "fotos": {"treino_pastas": n_treino, "teste_pastas": n_teste, "treino_com_mao": len(treino),
              "validacao_com_mao": len(validacao), "teste_com_mao": len(teste)},
    "teste": {"acuracia": round(float(acc), 4), "perda": round(float(perda), 4), **media},
    "por_letra": por_letra,
    "rotulos": rotulos,
    "matriz_confusao": matriz.tolist(),
    "historico": hist,
    "tempo_min": {"preparo": round(t_pre / 60, 1), "treino": round(t_treino / 60, 1)},
}
with open(os.path.join(a.saida, "resultado_task.json"), "w", encoding="utf8") as f:
    json.dump(res, f, ensure_ascii=False, indent=1)
shutil.rmtree(trab, ignore_errors=True)
print("\nPronto:", os.path.join(a.saida, "modelo", "gesture_recognizer.task"))


## 5. Treinar
As mensagens do TensorFlow ficam guardadas em `/content/resultado/log_treino.txt`; aqui aparecem só as linhas importantes e o progresso das épocas. Se der erro, a célula para e mostra o fim do log.

In [ ]:
import subprocess, re
cmd = ["/content/mm310/bin/python", "-u", "-W", "ignore", "/content/treinar_task.py",
       "--dados", DADOS, "--saida", "/content/resultado",
       "--epocas", str(EPOCAS), "--lote", str(LOTE), "--semente", str(SEMENTE)]
if ESPELHAR:
    cmd.append("--espelhar")
ruido = re.compile(r"^(W0000|I0000|E0000)|WARNING|warnings.warn|UserWarning|tensorflow/|^\s*$")
log_path = "/content/resultado/log_treino.txt"
os.makedirs("/content/resultado", exist_ok=True)
ultimas = []
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env={**os.environ, "MPLBACKEND": "Agg"})
    for linha in proc.stdout:
        log.write(linha)
        ultimas = (ultimas + [linha])[-60:]
        if not ruido.search(linha):
            print(linha, end="")
    proc.wait()
if proc.returncode != 0:
    print("\n===== FIM DO LOG =====\n" + "".join(ultimas))
    raise RuntimeError(f"O treino falhou (código {proc.returncode}). Copie as linhas acima e mande para o Claude. "
                       "Código -9 costuma ser falta de memória: tente ESPELHAR = False ou FRACAO menor.")

## 6. Resultados e gráficos

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

r = json.load(open("/content/resultado/resultado_task.json"))
print("Fotos:", r["fotos"])
print("Teste:", r["teste"])
print("Tempo (min):", r["tempo_min"])

h = r["historico"]
ep = range(1, len(h["loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, h["categorical_accuracy"], label="treino")
ax[0].plot(ep, h["val_categorical_accuracy"], label="validação")
ax[0].set_title("Acurácia por época"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(ep, h["loss"], label="treino")
ax[1].plot(ep, h["val_loss"], label="validação")
ax[1].set_title("Perda por época"); ax[1].set_xlabel("época"); ax[1].legend()
plt.tight_layout(); plt.savefig("/content/resultado/curvas_task.png", dpi=150); plt.show()

rot = r["rotulos"]
m = np.array(r["matriz_confusao"])
usar = [i for i, x in enumerate(rot) if x != "none"]
m = m[np.ix_(usar, usar)]
let = [rot[i] for i in usar]
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(m, cmap="Blues")
ax.set_xticks(range(len(let)), let); ax.set_yticks(range(len(let)), let)
ax.set_xlabel("letra prevista"); ax.set_ylabel("letra verdadeira")
ax.set_title(f"Matriz de confusão do .task no teste (acurácia {r['teste']['acuracia']:.3f})")
for i in range(len(let)):
    for j in range(len(let)):
        if m[i, j]:
            ax.text(j, i, m[i, j], ha="center", va="center", fontsize=7,
                    color="white" if m[i, j] > m.max() / 2 else "black")
plt.tight_layout(); plt.savefig("/content/resultado/matriz_confusao_task.png", dpi=150); plt.show()

## 7. Baixar

In [ ]:
import shutil
from google.colab import files
shutil.copy("/content/resultado/modelo/gesture_recognizer.task", "/content/resultado/gesture_recognizer.task")
!cd /content/resultado && rm -f /content/treino_task_libras.zip && zip -q /content/treino_task_libras.zip gesture_recognizer.task resultado_task.json lista_fotos.txt curvas_task.png matriz_confusao_task.png
files.download("/content/treino_task_libras.zip")